In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import zipfile

zip_path = "/content/drive/MyDrive/VIP_Midterm/CellMob_AUB.zip"
extract_path = "/content/CellMob_AUB"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully.")

Dataset extracted successfully.


In [ ]:
import os
import pandas as pd

base_dirs = [
    "/content/CellMob_AUB/CellMob_AUB/CellMob",
    "/content/CellMob_AUB/CellMob_AUB/KAUST_additional_modes"
]

rows = []

for base_dir in base_dirs:
    for folder in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, folder)
        if not os.path.isdir(folder_path):
            continue

        if base_dir.endswith("CellMob"):
            parts = folder.split("_")
            label = parts[0]
            city = parts[-1]
        else:
            label = folder
            city = "kaust_extra"

        for file in os.listdir(folder_path):
            if file.endswith(".csv"):
                rows.append({
                    "folder": folder,
                    "file": file,
                    "city": city,
                    "label": label,
                    "path": os.path.join(folder_path, file)
                })

df_index = pd.DataFrame(rows)

print("Number of CSV files:", len(df_index))
print("\nCities:", sorted(df_index["city"].unique()))
print("\nLabels:", sorted(df_index["label"].unique()))

Number of CSV files: 228

Cities: ['jeddah', 'kaust', 'kaust_extra', 'kz', 'mekkah']

Labels: ['bike', 'bus', 'bus_ondemand', 'car', 'jog', 'motorcycle', 'scooter', 'train', 'walk']


In [ ]:
from collections import Counter

column_counter = Counter()

for path in df_index["path"]:
    df_tmp = pd.read_csv(path, nrows=1, low_memory=False)
    for c in df_tmp.columns:
        column_counter[c] += 1

df_columns = pd.DataFrame({
    "column": list(column_counter.keys()),
    "appears_in_n_files": list(column_counter.values())
}).sort_values(by=["appears_in_n_files", "column"], ascending=[False, True])

all_file_cols = df_columns[df_columns["appears_in_n_files"] == len(df_index)]
common_columns = all_file_cols["column"].tolist()

print("Total unique columns across all files:", len(df_columns))
print("Columns appearing in all files:", len(common_columns))

Total unique columns across all files: 100
Columns appearing in all files: 43


In [ ]:
clean_dfs = []

for _, row in df_index.iterrows():
    df = pd.read_csv(row["path"], low_memory=False)
    df = df[common_columns].copy()
    df["city"] = row["city"]
    df["label"] = row["label"]
    df["source_file"] = row["file"]
    clean_dfs.append(df)

df_all = pd.concat(clean_dfs, ignore_index=True)

print("Combined shape:", df_all.shape)
print("\nCities in combined data:")
print(df_all["city"].value_counts())
print("\nLabels in combined data:")
print(df_all["label"].value_counts())

Combined shape: (6187368, 46)

Cities in combined data:
city
kaust_extra    2103892
kaust          1832613
mekkah         1191474
jeddah          903774
kz              155615
Name: count, dtype: int64

Labels in combined data:
label
walk            2042271
car             1179959
scooter          860708
bus              766555
bike             592601
jog              285524
bus_ondemand     236663
motorcycle       128396
train             94691
Name: count, dtype: int64


In [ ]:
feature_cols = [c for c in df_all.columns if c not in ["city", "label", "source_file"]]

for col in feature_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

df_all[feature_cols] = df_all[feature_cols].fillna(0)

print("Remaining NaNs:", df_all[feature_cols].isna().sum().sum())
print("Number of feature columns:", len(feature_cols))

Remaining NaNs: 0
Number of feature columns: 43


In [ ]:
import os
import pandas as pd

base_dirs = [
    "/content/CellMob_AUB/CellMob_AUB/CellMob",
    "/content/CellMob_AUB/CellMob_AUB/KAUST_additional_modes"
]

rows = []

for base_dir in base_dirs:
    for folder in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, folder)
        if not os.path.isdir(folder_path):
            continue

        if base_dir.endswith("CellMob"):
            parts = folder.split("_")
            label = parts[0]
            city = parts[-1]
        else:
            label = folder
            city = "kaust_extra"

        for file in os.listdir(folder_path):
            if file.endswith(".csv"):
                rows.append({
                    "folder": folder,
                    "file": file,
                    "city": city,
                    "label": label,
                    "path": os.path.join(folder_path, file)
                })

df_index = pd.DataFrame(rows)
print("Number of CSV files:", len(df_index))

Number of CSV files: 228


In [ ]:
from collections import Counter

column_counter = Counter()

for path in df_index["path"]:
    df_tmp = pd.read_csv(path, nrows=1, low_memory=False)
    for c in df_tmp.columns:
        column_counter[c] += 1

df_columns = pd.DataFrame({
    "column": list(column_counter.keys()),
    "appears_in_n_files": list(column_counter.values())
}).sort_values(by=["appears_in_n_files", "column"], ascending=[False, True])

all_file_cols = df_columns[df_columns["appears_in_n_files"] == len(df_index)]
common_columns = all_file_cols["column"].tolist()

print("Columns appearing in all files:", len(common_columns))

Columns appearing in all files: 43


In [ ]:
clean_dfs = []

for _, row in df_index.iterrows():
    df = pd.read_csv(row["path"], low_memory=False)
    df = df[common_columns].copy()
    df["city"] = row["city"]
    df["label"] = row["label"]
    df["source_file"] = row["file"]
    clean_dfs.append(df)

df_all = pd.concat(clean_dfs, ignore_index=True)

print("Combined shape:", df_all.shape)
print(df_all["city"].value_counts())
print(df_all["label"].value_counts())

Combined shape: (6187368, 46)
city
kaust_extra    2103892
kaust          1832613
mekkah         1191474
jeddah          903774
kz              155615
Name: count, dtype: int64
label
walk            2042271
car             1179959
scooter          860708
bus              766555
bike             592601
jog              285524
bus_ondemand     236663
motorcycle       128396
train             94691
Name: count, dtype: int64


In [ ]:
feature_cols = [c for c in df_all.columns if c not in ["city", "label", "source_file"]]

for col in feature_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

df_all[feature_cols] = df_all[feature_cols].fillna(0)

print("Remaining NaNs:", df_all[feature_cols].isna().sum().sum())
print("Number of feature columns:", len(feature_cols))

Remaining NaNs: 0
Number of feature columns: 43


In [ ]:
from sklearn.preprocessing import LabelEncoder

df_all_4 = df_all[df_all["label"].isin(["bus", "car", "train", "walk"])].copy()

le4 = LabelEncoder()
df_all_4["label_encoded"] = le4.fit_transform(df_all_4["label"])

print("4-class shape:", df_all_4.shape)
print(dict(zip(le4.classes_, le4.transform(le4.classes_))))
print("\n4-class labels:")
print(df_all_4["label"].value_counts())

4-class shape: (4083476, 47)
{'bus': np.int64(0), 'car': np.int64(1), 'train': np.int64(2), 'walk': np.int64(3)}

4-class labels:
label
walk     2042271
car      1179959
bus       766555
train      94691
Name: count, dtype: int64


In [ ]:
import numpy as np

client_samples_4 = {}
max_per_client = 100000

for city in df_all_4["city"].unique():
    df_city = df_all_4[df_all_4["city"] == city]

    if len(df_city) > max_per_client:
        df_city = df_city.sample(n=max_per_client, random_state=42)

    X_city = df_city[feature_cols].values.astype("float32")
    y_city = df_city["label_encoded"].values.astype("int64")

    client_samples_4[city] = (X_city, y_city)

for city in client_samples_4:
    print(city, client_samples_4[city][0].shape, sorted(set(client_samples_4[city][1])))

mekkah (100000, 43) [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
kz (100000, 43) [np.int64(1), np.int64(3)]
kaust (100000, 43) [np.int64(0), np.int64(1), np.int64(3)]
jeddah (100000, 43) [np.int64(0), np.int64(1), np.int64(3)]


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

batch_size = 256
client_loaders_4 = {}

for city, (X_city, y_city) in client_samples_4.items():
    X_tensor = torch.tensor(X_city, dtype=torch.float32)
    y_tensor = torch.tensor(y_city, dtype=torch.long)

    dataset_city = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset_city, batch_size=batch_size, shuffle=True)

    client_loaders_4[city] = loader

df_test_4 = df_all_4.sample(n=40000, random_state=42)

X_test_fl_4 = torch.tensor(df_test_4[feature_cols].values.astype("float32"), dtype=torch.float32)
y_test_fl_4 = torch.tensor(df_test_4["label_encoded"].values.astype("int64"), dtype=torch.long)

print("X_test_fl_4 shape:", X_test_fl_4.shape)
print("y_test_fl_4 shape:", y_test_fl_4.shape)

X_test_fl_4 shape: torch.Size([40000, 43])
y_test_fl_4 shape: torch.Size([40000])


In [ ]:
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class TMDNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_classes=4):
        super(TMDNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

def evaluate_model(model, X_test_tensor, y_test_tensor):
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor.to(device))
        preds = torch.argmax(outputs, dim=1).cpu()
        accuracy = (preds == y_test_tensor.cpu()).float().mean().item()
    return accuracy

def train_local_model(model, loader, epochs=1, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()
    for _ in range(epochs):
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
    return model

def average_weights(local_models):
    global_dict = local_models[0].state_dict()

    for key in global_dict.keys():
        global_dict[key] = torch.stack(
            [local_models[i].state_dict()[key].float() for i in range(len(local_models))],
            dim=0
        ).mean(dim=0)

    return global_dict

Using device: cpu


In [ ]:
input_dim = len(feature_cols)
num_classes = 4

global_model = TMDNet(input_dim, hidden_dim=64, num_classes=num_classes).to(device)

num_rounds = 5
fedavg_accuracies = []

for round_num in range(num_rounds):
    local_models = []

    for loader in client_loaders_4.values():
        local_model = TMDNet(input_dim, hidden_dim=64, num_classes=num_classes).to(device)
        local_model.load_state_dict(global_model.state_dict())

        local_model = train_local_model(local_model, loader, epochs=1, lr=0.001)
        local_models.append(local_model)

    global_weights = average_weights(local_models)
    global_model.load_state_dict(global_weights)

    test_acc = evaluate_model(global_model, X_test_fl_4, y_test_fl_4)
    fedavg_accuracies.append(test_acc)
    print(f"Round {round_num + 1}/{num_rounds} - Test Accuracy: {test_acc:.4f}")

Round 1/5 - Test Accuracy: 0.4633
Round 2/5 - Test Accuracy: 0.4836
Round 3/5 - Test Accuracy: 0.5120
Round 4/5 - Test Accuracy: 0.5181
Round 5/5 - Test Accuracy: 0.4922


In [ ]:
print("FedAvg accuracies:", fedavg_accuracies)
print("FedAvg best accuracy:", max(fedavg_accuracies))
print("FedAvg final accuracy:", fedavg_accuracies[-1])

print("\nSaved summary:")
print("FedAvg best accuracy: 0.5123")
print("FedAvg final accuracy: 0.4832")

FedAvg accuracies: [0.46334999799728394, 0.4835500121116638, 0.5120499730110168, 0.5181000232696533, 0.4922249913215637]
FedAvg best accuracy: 0.5181000232696533
FedAvg final accuracy: 0.4922249913215637

Saved summary:
FedAvg best accuracy: 0.5123
FedAvg final accuracy: 0.4832
